<a href="https://colab.research.google.com/github/marco-dev-pinheiro/ETL/blob/main/extrator_padronizador_de_observa%C3%A7oes_notas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Logica do ETL

* Ordem de execução

1.   Instalação das dependencias
2.   Importações de modulos
3.   Extração de dados preveamente mapeados do pdf
4.   Função de manipilação de dados e entrega de dados
5.   Interface Gradio


In [1]:
# ============================
# Instalação de Dependências
# ============================
!pip install gradio pypdf -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 6.9 MB/s eta 0:00:00


In [2]:
# ============================
# Importações
# ============================
import gradio as gr
import re
import os
from pypdf import PdfReader


In [3]:
# ============================
# Classe de Extração
# ============================
class ReservaExtractor:
    def __init__(self, texto_bruto):
        self.texto = texto_bruto

    def _capturar(self, patterns, multi_line=False):
        """Captura informações usando expressões regulares."""
        if isinstance(patterns, str):
            patterns = [patterns]
        flags = re.IGNORECASE
        if multi_line:
            flags |= re.DOTALL
        for pattern in patterns:
            match = re.search(pattern, self.texto, flags)
            if match:
                return match.group(1).strip()
        return "Não encontrado"

    def extrair(self, num_apto=""):
        """Extrai dados relevantes do texto do voucher ou extrato."""

        # Número da reserva
        codigo_cmnet = self._capturar([
            r"RESERVA INTEGRADA cmnet COM O CÓDIGO\s*(\d+)",
            r"Reserva[:\s]+(\d+)"
        ])
        reserva_final = codigo_cmnet if codigo_cmnet != "Não encontrado" else "Não encontrado"

        # Confirmação
        confirmacao = self._capturar(r"Confirmação da Reserva[:\s]+(\d+)")
        confirmacao_str = f"Reserva Confirmação: {confirmacao}" if confirmacao != "Não encontrado" else "Reserva Não encontrado"

        # Nome do hóspede
        hospede_raw = self._capturar([
            r"Hóspedes\s*\([^)]+\):\s*\(([^)]+)\)",
            r"H[OÓoó]SPEDE[:\s]+([A-Z\sÁÉÍÓÚÂÊÔÇ]+?)(?=\n|OS:|Empresa|Grupo|\-|#)",
            r"Nome[:\s]+([A-Z\sÁÉÍÓÚÂÊÔÇ]+?)(?=\n|Pendente|Caf[eé]|Empresa|Grupo|\-|#)"
        ])
        hospede = hospede_raw.replace('/', ' ') if hospede_raw != "Não encontrado" else "Não encontrado"
        hospede = re.sub(r'\s+', ' ', hospede).strip()

        # Datas
        checkin = self._capturar([r"Check[- ]in[:\s]+(\d{2}/\d{2}/\d{4})", r"Chegada[:\s]+(\d{2}/\d{2}/\d{4})"])
        checkout = self._capturar([r"Check[- ]out[:\s]+(\d{2}/\d{2}/\d{4})", r"Partida[:\s]+(\d{2}/\d{2}/\d{4})"])

        # Categoria do quarto
        cat_isolada = self._capturar(r"Categoria[:\s]+([^\n]+)")
        if cat_isolada == "Não encontrado":
            match_hotel = re.search(r"LAGHETTO\s+([A-Z]+)\s+HIGIENOPOLIS", self.texto, re.IGNORECASE)
            cat_isolada = match_hotel.group(1).strip() if match_hotel else "Não informado"

        # Tipo do quarto
        tipo_isolado = self._capturar([r"Tipo Apto[:\s]+([^\n\s]+)", r"TRF\s+([A-Z]+)"])
        if tipo_isolado == "Não encontrado":
            texto_upper = self.texto.upper()
            if "SINGLE" in texto_upper or "SGL" in texto_upper:
                tipo_isolado = "SGL"
            elif "DOUBLE" in texto_upper or "TWIN" in texto_upper or "DBL" in texto_upper:
                tipo_isolado = "DBL"
            else:
                tipo_isolado = "Não informado"

        categoria_apto = f"Categoria: {cat_isolada} Tipo Apto: {tipo_isolado}"

        # OS / Observação
        os_match = self._capturar([r"OS[:\s]+(\d+)", r"Observação[:\s]+(?:OS\s*)?(\d+)", r"#(\d+)"])
        os_servico = os_match if os_match != "Não encontrado" else "Não encontrado"

        # Valores extras / serviço
        servico_valor = self._capturar([
            r"SERVICO\s*\n*\s*BRL\s*([\d,.]+)",
            r"SERVICO\s+BRL\s+([\d,.]+)",
            r"ISS\s*\|\s*([\d,.]+)"
        ])

        # Valor da diária
        diaria_raw = self._capturar([
            r"Valor médio da Diária para o período[:\s]*BRL\s*([\d,.]+)",
            r"Diária[:\s]*BRL\s*([\d,.]+)",
            r"Tarifa[:\s]*BRL\s*([\d,.]+)"
        ])
        diaria_valor = re.sub(r',00$', '', diaria_raw.strip()) if diaria_raw != "Não encontrado" else "Não encontrado"

        # Montagem da resposta
        partes = [
            f"Numero de reserva {reserva_final}",
            confirmacao_str,
            f"Check-In: {checkin} Check-Out: {checkout}",
            f"Hóspedes: {hospede}",
            f"{categoria_apto}"
        ]

        if num_apto.strip():
            partes.append(f"Apto: {num_apto.strip()}")
        if os_servico != "Não encontrado":
            partes.append(f"OS {os_servico}")
        if servico_valor != "Não encontrado":
            partes.append(f"SERVICO {servico_valor}")
        if diaria_valor != "Não encontrado":
            partes.append(f"VALOR DIARIA {diaria_valor}")

        return " // ".join(partes) + " //"




In [4]:
# ============================
# Função de Processamento
# ============================
def processar_voucher(arquivo, num_apto):
    """Processa o arquivo PDF ou TXT e retorna os dados extraídos."""
    if arquivo is None:
        return "Por favor, realize o upload de um voucher (PDF ou TXT)."

    texto_extraido = ""
    caminho_arquivo = arquivo.name

    try:
        if caminho_arquivo.lower().endswith('.pdf'):
            reader = PdfReader(caminho_arquivo)
            for page in reader.pages:
                texto_extraido += page.extract_text() + "\n"
        elif caminho_arquivo.lower().endswith('.txt'):
            with open(caminho_arquivo, 'r', encoding='utf-8') as f:
                texto_extraido = f.read()
        else:
            return "Extensão inválida. Envie arquivos em formato PDF ou TXT."

        extractor = ReservaExtractor(texto_extraido)
        return extractor.extrair(num_apto)

    except Exception as e:
        return f"Falha no processamento do documento: {str(e)}"


# Interface Gradio

In [5]:
# ============================
# Interface Gradio
# ============================
with gr.Blocks(title="Extrator Híbrido de Reservas") as app:
    gr.Markdown("### Extrator Inteligente Unificado (Vouchers Voetur & Extratos de Hotéis)")

    with gr.Row():
        with gr.Column():
            file_entrada = gr.File(label="Arraste seu Arquivo aqui (PDF ou TXT)", file_types=[".pdf", ".txt"])
            txt_apto = gr.Textbox(label="Número do Apartamento (Manual)", placeholder="Ex: 1012")

        txt_saida = gr.Textbox(label="Saída Formatada Padrão", lines=6)

    btn_processar = gr.Button("Processar Documento", variant="primary")
    btn_processar.click(fn=processar_voucher, inputs=[file_entrada, txt_apto], outputs=txt_saida)

app.launch(inline=True, share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://41cbaac6a405b5a803.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
